# Check your installation

Run both cells below. This checks that the environment you are using right now
can run the course notebooks.

Before running this, follow the setup steps in the README: create a Python 3.12
environment and install the requirements with `pip install -r requirements.txt`.

**Use the same kernel you will use for the course notebooks.** That is the point
of running this as a notebook rather than as a script: it checks the kernel
Jupyter actually gives you, not whichever Python your terminal happens to use.

Nothing is installed and no internet connection is needed. Anything that fails
prints the command that fixes it.


## The checks

You can collapse this cell, it is just the checking code.

In [3]:
from __future__ import annotations

import importlib
import importlib.metadata as md
import os
import re
import sys
from pathlib import Path

REQUIRED_PYTHON = (3, 12)
def _find_repository_root() -> Path:
    """Walk upwards until we find the folder holding requirements.txt."""
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    return here


ROOT = _find_repository_root()

# Distribution name on PyPI -> the name you import in Python.
IMPORT_NAME = {
    "scikit-learn": "sklearn",
    "scikit-learn-extra": "sklearn_extra",
    "umap-learn": "umap",
    "cca-zoo": "cca_zoo",
    "jupyter": "jupyter",
}

EXPECTED_NOTEBOOKS = [
    "data_preprocessing.ipynb",
    "1_feature_transformation.ipynb",
    "2_autoencoders.ipynb",
    "3_feature_aggregation.ipynb",
    "4_feature_selection.ipynb",
    "5_stability_optimization.ipynb",
]

EXPECTED_DATA = [
    "GSE45774_rpkm_all.txt",
    "goslim_to_genes.txt",
    "tomato_with_targets.txt",
]

_TTY = sys.stdout.isatty()
failed = 0
warned = 0


def _c(code: str, text: str) -> str:
    return f"\033[{code}m{text}\033[0m" if _TTY else text


def head(text: str) -> None:
    print(f"\n{_c('1', text)}")


def ok(text: str) -> None:
    print(f"  {_c('32', 'OK')}   {text}")


def bad(text: str, fix: str = "") -> None:
    global failed
    print(f"  {_c('31', 'FAIL')} {text}")
    if fix:
        print(f"       -> {fix}")
    failed += 1


def warn(text: str, fix: str = "") -> None:
    global warned
    print(f"  {_c('33', 'WARN')} {text}")
    if fix:
        print(f"       -> {fix}")
    warned += 1


def check_python() -> None:
    head("Python")
    v = sys.version_info
    want = f"{REQUIRED_PYTHON[0]}.{REQUIRED_PYTHON[1]}"
    if (v.major, v.minor) == REQUIRED_PYTHON:
        ok(f"Python {v.major}.{v.minor}.{v.micro}")
    else:
        bad(
            f"Python {v.major}.{v.minor}.{v.micro}, but this course needs {want}",
            f"conda create -n dimred python={want} && conda activate dimred, "
            f"or python{want} -m venv .venv && source .venv/bin/activate",
        )
    # sys.prefix differs from base_prefix inside a venv, activated or not.
    if sys.prefix != sys.base_prefix:
        ok(f"isolated environment in use: {Path(sys.prefix).name}")
    elif os.environ.get("CONDA_DEFAULT_ENV"):
        ok(f"conda environment active: {os.environ['CONDA_DEFAULT_ENV']}")
    else:
        warn("running against the system Python, not an isolated environment",
             "create an environment first, see the README")


def check_packages() -> None:
    head("Packages from requirements.txt")
    req = ROOT / "requirements.txt"
    if not req.is_file():
        bad("requirements.txt not found", "run this from the repository root")
        return
    pattern = re.compile(r"^\s*([A-Za-z0-9._-]+)\s*==\s*([^\s#]+)")
    for line in req.read_text().splitlines():
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        m = pattern.match(line)
        if not m:
            warn(f"could not read this requirement line: {line.strip()}")
            continue
        dist, want = m.group(1), m.group(2)
        try:
            have = md.version(dist)
        except md.PackageNotFoundError:
            bad(f"{dist} is not installed",
                "pip install -r requirements.txt")
            continue
        if have == want:
            ok(f"{dist} {have}")
        else:
            warn(f"{dist} {have}, but requirements.txt pins {want}",
                 f"pip install '{dist}=={want}'")


def check_imports() -> None:
    head("Imports used by the notebooks")
    for module in ["numpy", "pandas", "sklearn", "sklearn_extra", "umap",
                   "cca_zoo", "ReliefF", "matplotlib", "seaborn"]:
        try:
            importlib.import_module(module)
            ok(f"{module}")
        except Exception as exc:
            bad(f"{module} does not import: {type(exc).__name__}: {exc}",
                "pip install -r requirements.txt")


def check_distutils() -> None:
    """scikit-learn-extra imports distutils, which Python 3.12 removed."""
    head("distutils (needed by scikit-learn-extra on Python 3.12)")
    if sys.version_info < (3, 12):
        ok("Python below 3.12 still ships distutils")
        return
    try:
        import setuptools  # noqa: F401
        ok(f"setuptools {md.version('setuptools')} provides the distutils shim")
    except ModuleNotFoundError:
        bad("setuptools is missing, so scikit-learn-extra will crash on first use",
            "pip install -r requirements.txt   # setuptools is listed there")


def check_pandas_behaviour() -> None:
    """The notebooks use drop(columns=..., axis=1), which pandas 3 rejects."""
    head("pandas compatibility")
    try:
        import pandas as pd
    except Exception as exc:
        bad(f"pandas does not import: {exc}")
        return
    try:
        frame = pd.DataFrame({"a": [1, 2], "b": [3, 4]})
        frame.drop(columns=["b"], axis=1)
        ok(f"pandas {pd.__version__} accepts the drop() call the notebooks use")
    except Exception as exc:
        bad(f"pandas {pd.__version__} rejects drop(columns=..., axis=1): {exc}",
            "pip install -r requirements.txt   # pandas is pinned below 3.0")


def check_functional() -> None:
    head("Functional checks (nothing is downloaded)")
    try:
        from sklearn.ensemble import RandomForestClassifier
        clf = RandomForestClassifier(n_estimators=4, random_state=0)
        clf.fit([[0, 0], [1, 1], [0, 1], [1, 0]], [0, 1, 0, 1])
        ok("scikit-learn trains a small model")
    except Exception as exc:
        bad(f"scikit-learn failed: {type(exc).__name__}: {exc}")

    try:
        import numpy as np
        from sklearn_extra.cluster import KMedoids
        points = np.array([[0, 0], [0, 1], [1, 0], [8, 8], [8, 9], [9, 8]], float)
        KMedoids(n_clusters=2, random_state=0).fit(points)
        ok("scikit-learn-extra runs KMedoids")
    except Exception as exc:
        bad(f"scikit-learn-extra failed: {type(exc).__name__}: {exc}",
            "check the distutils result above")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        figure, axes = plt.subplots()
        axes.plot([0, 1], [0, 1])
        figure.canvas.draw()
        plt.close(figure)
        ok("matplotlib renders a figure")
    except Exception as exc:
        bad(f"matplotlib failed: {type(exc).__name__}: {exc}")

    try:
        import umap  # noqa: F401
        ok("umap imports, numba compiled successfully")
    except Exception as exc:
        bad(f"umap failed: {type(exc).__name__}: {exc}")

    # 2_autoencoders.ipynb sets this itself; do the same so the check matches.
    os.environ.setdefault("KERAS_BACKEND", "torch")
    try:
        import keras
        ok(f"keras {keras.__version__} on the {keras.backend.backend()} backend")
    except Exception as exc:
        bad(f"keras failed: {type(exc).__name__}: {exc}",
            "keras needs torch as its backend, which requirements.txt installs")


def check_repository() -> None:
    head("Repository contents")
    notebooks = ROOT / "notebooks"
    for name in EXPECTED_NOTEBOOKS:
        if (notebooks / name).is_file():
            ok(f"notebooks/{name}")
        else:
            bad(f"notebooks/{name} is missing", "the clone looks incomplete")
    if (notebooks / "helpers.py").is_file():
        ok("notebooks/helpers.py")
    else:
        bad("notebooks/helpers.py is missing", "the clone looks incomplete")
    for name in EXPECTED_DATA:
        path = ROOT / "data" / name
        if path.is_file():
            ok(f"data/{name} ({path.stat().st_size / 1e6:.1f} MB)")
        elif name == "tomato_with_targets.txt":
            warn("data/tomato_with_targets.txt is missing",
                 "run notebooks/data_preprocessing.ipynb to generate it")
        else:
            bad(f"data/{name} is missing", "the clone looks incomplete")


def check_jupyter() -> None:
    head("Jupyter")
    try:
        import ipykernel  # noqa: F401
        ok("ipykernel is available, so this environment can be used as a kernel")
    except Exception:
        bad("ipykernel is missing, the notebooks cannot run in this environment",
            "pip install -r requirements.txt")


def main() -> int:
    print(_c("1", "DimRed course, installation check"))
    check_python()
    check_packages()
    check_imports()
    check_distutils()
    check_pandas_behaviour()
    check_functional()
    check_repository()
    check_jupyter()

    print()
    if failed:
        print(_c("31", f"{failed} check(s) failed") +
              (f", {warned} warning(s)" if warned else "") +
              ". Fix the FAIL lines above, then run this again.")
        return 1
    if warned:
        print(_c("32", "All critical checks passed") +
              f", {warned} warning(s). You are ready for the course.")
        return 0
    print(_c("32", "All checks passed. You are ready for the course."))
    return 0


## Run them

In [4]:
exit_code = main()


DimRed course, installation check

Python
  OK   Python 3.12.13
  OK   isolated environment in use: .venv

Packages from requirements.txt
  OK   matplotlib 3.11.1
  OK   numpy 1.26.4
  OK   pandas 2.3.3
  OK   scikit-learn 1.9.0
  OK   scikit-learn-extra 0.3.0
  OK   setuptools 84.0.0
  OK   umap-learn 0.5.12
  OK   cca-zoo 3.1.1
  OK   seaborn 0.13.2
  OK   ReliefF 0.1.2
  OK   keras 3.15.1
  OK   torch 2.13.0
  OK   jupyter 1.1.1

Imports used by the notebooks
  OK   numpy
  OK   pandas
  OK   sklearn
  OK   sklearn_extra
  OK   umap
  OK   cca_zoo
  OK   ReliefF
  OK   matplotlib
  OK   seaborn

distutils (needed by scikit-learn-extra on Python 3.12)
  OK   setuptools 84.0.0 provides the distutils shim

pandas compatibility
  OK   pandas 2.3.3 accepts the drop() call the notebooks use

Functional checks (nothing is downloaded)
  OK   scikit-learn trains a small model
  OK   scikit-learn-extra runs KMedoids
  OK   matplotlib renders a figure
  OK   umap imports, numba compiled succes